
# Loan Default Prediction: from raw data to a model you can deploy

## The story
You work in the **credit-risk team of a bank**. Today 1,000 people asked for a loan.
Some will pay it back. Some will not. Can we use past loans to **estimate the chance that a new applicant will default**?

* **Target:** `loan_status` &rarr; `1` = the person **defaulted**, `0` = the person **repaid**
* **Our goal:** predict the *probability* of default for every applicant (not just yes / no)

## Roadmap
| Step | What we do | Bank question |
|---|---|---|
| 1 | Set up | Get the tools ready |
| 2 | Get the data | What do we know about past borrowers? |
| 3 | Clean the data | Can we trust it? |
| 4 | Explore (EDA) | What patterns do we see? |
| 5 | Prepare the data | How do we turn it into numbers a model understands? |
| 6 | Split + dummy baseline | What is the *minimum* score we must beat? |
| 7 | Baseline model | How good is a simple model? |
| 8 | Feature importance | What does the bank look at most? |
| 9 | LightGBM | Can a stronger model do better? |
| 10 | Final model + submission | Train on everything, predict the test set |
| 11 | Save and use the model | Score one new applicant |

> **One golden rule for the whole notebook:** *anything the model learns (medians, caps, scalers, encoders) must be learned from the training data only.*
> Think of it like an exam: you study from the textbook (train) and you never peek at the exam paper (test).

In [1]:
import sys, numpy
print(sys.executable)

C:\Users\nuhaa\loan-default-project\.venv\Scripts\python.exe


## 1. Setup

Install once (if needed): `pip install pandas numpy scikit-learn lightgbm statsmodels seaborn matplotlib joblib`

Put these four files **in the same folder as this notebook** (or change `DATA_DIR`):
`train.csv`, `test.csv`, `credit_risk_dataset.csv`, `sample_submission.csv`

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

%matplotlib inline

assert sys.version_info >= (3, 9), "Please use Python 3.9 or later"
sk_version = tuple(int(p) for p in sklearn.__version__.split(".")[:2])
assert sk_version >= (1, 2), "Please upgrade scikit-learn:  pip install -U scikit-learn"
assert tuple(int(p) for p in sns.__version__.split(".")[:2]) >= (0, 13), "Please upgrade seaborn:  pip install -U seaborn"

sns.set_theme(style="whitegrid")
plt.rcParams.update({"figure.dpi": 100, "axes.labelsize": 12})
pd.options.display.float_format = "{:.3f}".format
np.set_printoptions(precision=3, suppress=True)

RANDOM_STATE = 42               # same number = same results every time
DATA_DIR = Path("data")            # folder that contains the four CSV files

ModuleNotFoundError: No module named 'numpy'

## 2. Get the data

| File | What it is | Analogy |
|---|---|---|
| `train.csv` | Past loans **with** the answer (`loan_status`) | Practice questions with answers |
| `test.csv` | New loans **without** the answer | The exam paper |
| `credit_risk_dataset.csv` | The original real-world dataset that `train.csv` was generated from | Extra practice questions |
| `sample_submission.csv` | The format we must use to send predictions | The answer sheet |

In [ ]:
train_raw = pd.read_csv(DATA_DIR / "train.csv")
test_raw = pd.read_csv(DATA_DIR / "test.csv")
original_raw = pd.read_csv(DATA_DIR / "credit_risk_dataset.csv")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

print("train:", train_raw.shape, "| test:", test_raw.shape,
      "| original:", original_raw.shape, "| sample submission:", sample_submission.shape)

# Safety check: the answer sheet must be in the same order as the exam paper
assert (sample_submission["id"].values == test_raw["id"].values).all(), "ids do not match!"

# The only column train has that the original does not is `id`
print("Columns only in train:", set(train_raw.columns) - set(original_raw.columns))

In [ ]:
train_raw.head()

### The columns (features)

| Column | Meaning | Type |
|---|---|---|
| `person_age` | Applicant's age | number |
| `person_income` | Yearly income | number |
| `person_home_ownership` | RENT / MORTGAGE / OWN / OTHER | category (no order) |
| `person_emp_length` | Years in the current job | number |
| `loan_intent` | Why they want the loan (education, medical, ...) | category (no order) |
| `loan_grade` | The bank's risk grade, **A (safest) to G (riskiest)** | category (has an **order**) |
| `loan_amnt` | Loan amount | number |
| `loan_int_rate` | Interest rate | number |
| `loan_percent_income` | Loan amount divided by income | number |
| `cb_person_default_on_file` | Did this person default before? (Y / N) | yes / no |
| `cb_person_cred_hist_length` | Years of credit history | number |
| **`loan_status`** | **TARGET:** 1 = defaulted, 0 = repaid | 0 / 1 |

> 🤔 **Think first:** if you were the banker, which three columns would you check before lending money? We will compare your guess with what the model finds in Step 8.

## 3. Clean the data

### 3.1 Combine train + original
More examples usually means a better model. We stack the two files on top of each other.
We also remove `id`: it is just a row number, it says nothing about repayment.
We keep the test ids because we need them for the submission file later.

In [ ]:
test_ids = test_raw["id"]                  # keep for the submission file
test = test_raw.drop(columns="id")

train = train_raw.drop(columns="id")
data = pd.concat([train, original_raw], axis=0, ignore_index=True)
print("Combined shape:", data.shape)

### 3.2 Duplicates
A duplicate row is like a student written twice in the attendance sheet: the model "learns" that person twice and becomes biased towards them.

In [ ]:
n_dup = data.duplicated().sum()
print(f"Duplicate rows: {n_dup} ({n_dup / len(data):.2%} of the data)")

data = data.drop_duplicates().reset_index(drop=True)
print("Shape after removing duplicates:", data.shape)

### 3.3 Missing values
How many are missing, and where?

In [ ]:
missing = data.isna().sum().to_frame("missing")
missing["percent"] = missing["missing"] / len(data) * 100
missing[missing["missing"] > 0]

Only `person_emp_length` and `loan_int_rate` have gaps. We will fill them in Step 5.
First, let us separate the features (`X`) from the answer (`y`).

In [ ]:
X = data.drop(columns="loan_status")
y = data["loan_status"]
print("Features:", X.shape, "| Target:", y.shape)

## 4. Explore the data (EDA)

EDA is **detective work**: we do not trust the data yet, so we question it.

### 4.1 How many people default?
This is the first thing to check, because it decides which metrics we can trust later.

In [ ]:
rate = y.mean()
print(f"Default rate: {rate:.1%}   |   Repaid: {1 - rate:.1%}")

fig, ax = plt.subplots(figsize=(4.5, 3.5))
sns.countplot(x=y, hue=y, palette="coolwarm", legend=False, ax=ax)
ax.set_xticks([0, 1], ["repaid (0)", "defaulted (1)"])
ax.set_title("Target balance")
plt.show()

print(f"\nA 'lazy' model that always says 'will repay' is correct {1 - rate:.1%} of the time, "
      "and catches zero defaulters.")

> 💡 **Why this matters:** the classes are **imbalanced** (far more people repay than default).
> So **accuracy can fool us**. Later we will use ROC-AUC, precision and recall.

### 4.2 Outliers: values that look wrong

In [ ]:
num_cols = ["person_age", "person_income", "person_emp_length", "loan_amnt",
            "loan_int_rate", "loan_percent_income", "cb_person_cred_hist_length"]

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, col in zip(axes.ravel(), num_cols):
    sns.boxplot(y=X[col], ax=ax, color="#8fb8de")
    ax.set_title(col)
axes.ravel()[-1].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
suspicious = pd.Series({
    "age above 100": (X["person_age"] > 100).sum(),
    "employment length above 50 years": (X["person_emp_length"] > 50).sum(),
    "employed longer than (age - 14) years": (X["person_emp_length"] > X["person_age"] - 14).sum(),
    "income above 1,000,000": (X["person_income"] > 1_000_000).sum(),
}, name="rows")
suspicious.to_frame()

> 🤔 **Ask yourself:** an age of 123 or 123 years of work. Is that a real person or a typing mistake?
> These are almost certainly **data-entry errors**. We cannot delete them from the *test* set (we must predict every row), so in Step 5 we **cap** them instead.

### 4.3 Which numbers move together? (correlation)
Correlation goes from -1 to +1. Near 0 means "no straight-line relationship".

**First, how does each number relate to the target?**

In [ ]:
df = pd.concat([X, y], axis=1)      # features + target together, for plotting

corr_with_target = df[num_cols + ["loan_status"]].corr()["loan_status"].drop("loan_status").sort_values()

fig, ax = plt.subplots(figsize=(7, 3.5))
corr_with_target.plot.barh(color=np.where(corr_with_target > 0, "#d9534f", "#4a78c2"), ax=ax)
ax.set_title("Correlation with loan_status (red = more default)")
ax.axvline(0, color="black", lw=0.8)
plt.show()

**Second, which features are copies of each other?** (two friends telling you the same story)

In [ ]:
corr = X[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
            vmin=-1, vmax=1, center=0, square=True, linewidths=1, ax=ax)
ax.set_title("Correlation between features")
plt.show()

pairs = corr.where(~mask).stack().dropna()
print("Pairs with |correlation| > 0.65:")
print(pairs[pairs.abs() > 0.65].to_string())

`person_age` and `cb_person_cred_hist_length` are strongly linked: older people have had more years to build a credit history.
`loan_amnt` and `loan_percent_income` are also related (a bigger loan is a bigger share of income), but less strongly.
We deal with this in Step 5.

### 4.4 Shape of each feature

In [ ]:
# Clipped at the 99.5th percentile ONLY for this picture, so a few giant values do not squash the chart
to_plot = X[num_cols].clip(upper=X[num_cols].quantile(0.995), axis=1)
to_plot.hist(bins=40, figsize=(16, 8), layout=(2, 4), color="#8fb8de", edgecolor="white")
plt.suptitle("Distribution of numeric features", y=1.02)
plt.tight_layout()
plt.show()

### 4.5 Who defaults? (the most useful charts)

**Important:** we plot the **default rate** (share of each group that defaulted), not raw counts.
A group can have many defaulters just because it is a big group. The rate tells us how *risky* the group really is.
The number `n` on each bar shows the group size.

In [ ]:
def plot_default_rate(data, col, ax, order=None):
    g = data.groupby(col)["loan_status"].agg(rate="mean", n="size")
    g = g.loc[order] if order is not None else g.sort_values("rate", ascending=False)
    ax.bar(g.index.astype(str), g["rate"], color="#4a78c2", edgecolor="black")
    ax.axhline(data["loan_status"].mean(), ls="--", color="gray", label="overall rate")
    for i, (r, n) in enumerate(zip(g["rate"], g["n"])):
        ax.text(i, r + 0.01, f"n={n:,}", ha="center", fontsize=8)
    ax.set_title(f"Default rate by {col}")
    ax.set_ylabel("share who defaulted")
    ax.set_ylim(0, min(1.05, g["rate"].max() + 0.12))
    ax.tick_params(axis="x", rotation=30)
    ax.legend()

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
plot_default_rate(df, "loan_grade", axes[0, 0], order=sorted(df["loan_grade"].unique()))
plot_default_rate(df, "person_home_ownership", axes[0, 1])
plot_default_rate(df, "loan_intent", axes[1, 0])
plot_default_rate(df, "cb_person_default_on_file", axes[1, 1])
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, col in zip(axes, ["loan_percent_income", "loan_int_rate", "person_income", "cb_person_cred_hist_length"]):
    sns.boxplot(data=df, x="loan_status", y=col, hue="loan_status", palette="coolwarm",
                legend=False, showfliers=False, ax=ax)
    ax.set_title(f"{col} by loan_status")
plt.tight_layout()
plt.show()

> 🤔 **Pause and discuss:** look at each chart *before* reading our summary.
> Which group is riskiest? Does the grade chart make sense for a bank?

### 4.6 Quick facts and what we learned

In [ ]:
print(f"Default rate overall: {y.mean():.1%}")
print("\nDefault rate by loan grade:")
print(df.groupby("loan_grade")["loan_status"].agg(["mean", "size"]).rename(columns={"mean": "default_rate", "size": "n"}).to_string())
print("\nShare of applicants by home ownership:")
print(X["person_home_ownership"].value_counts(normalize=True).round(3).to_string())
print("\nMost common loan purposes:")
print(X["loan_intent"].value_counts(normalize=True).round(3).to_string())

### EDA summary

**Data problems found &rarr; what we will do about them**

| Problem | Fix (Step 5) |
|---|---|
| Missing values in `person_emp_length` and `loan_int_rate` | Fill with the **median** |
| Impossible values (age or employment length above 100) | **Cap** employment length at 50 years; age is dropped |
| A few giant incomes | **Cap** income at the 99.5th percentile |
| `person_age` is almost a copy of `cb_person_cred_hist_length` | **Drop** `person_age` |
| Text columns | **Encode** them as numbers |
| Features on very different scales | **Scale** them |
| Duplicate rows | Already removed |

**Business insights**
* Only a minority of loans default, so **accuracy is not enough** as a metric.
* **Grade matters a lot:** the default rate climbs with the grade and jumps clearly from grade D upwards. Always read the *rate*, not the bar height: grades E, F and G have few loans but the highest rates.
* Borrowers who **rent**, who take a **large loan compared to their income**, or who pay a **high interest rate** default more often (see the charts above).
* A previous default on file is a warning sign (check the chart).

## 5. Prepare the data

Models only understand clean numbers. We will build a **recipe** (a scikit-learn *Pipeline*) with these ingredients:

| # | Ingredient | Analogy |
|---|---|---|
| 5.1 | Fill missing values | Fill the blanks in a form |
| 5.2 | Cap outliers | Put a ceiling on impossible values |
| 5.3 | Drop redundant features | Do not listen to two friends telling the same story |
| 5.4 | Create new features | Give the detective new clues |
| 5.5 | Encode categories | Translate words to numbers |
| 5.6 | Scale numbers | Compare kilometres with kilometres, not with centimetres |

### Why a Pipeline?
Every step **learns something from the data** (a median, a cap, a mean, the categories).
If we learn it from the *whole* dataset, information from the validation or test rows leaks into training, and our scores look better than they really are.
A Pipeline lets us call **`fit`** on training data only (*learn*), then **`transform`** on any data (*apply what was learned*).
Same idea as the exam: study from the textbook, then answer the exam with what you learned.

### 5.1 Missing values: median or mean?

In [ ]:
# Mini demo: 4 people earn 3 and one billionaire joins the room
demo = pd.Series([3, 3, 3, 3, 1000])
print("mean  :", demo.mean(), "  <- pulled up by one extreme value")
print("median:", demo.median(), "  <- tells the truth about a typical person")

In [ ]:
X[["person_emp_length", "loan_int_rate"]].agg(["mean", "median", "skew"])

Both columns are **right-skewed** (positive skew: a few big values pull the mean up), so we fill with the **median**.
`SimpleImputer` learns the median from the training rows and reuses it for new rows.

In [ ]:
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

impute_cols = ["person_emp_length", "loan_int_rate"]

imputer = ColumnTransformer(
    [("median_fill", SimpleImputer(strategy="median"), impute_cols)],
    remainder="passthrough",              # keep all the other columns untouched
    verbose_feature_names_out=False,
).set_output(transform="pandas")          # keep column names so later steps can find them

### 5.2 Outliers: cap them
**Capping** (also called *winsorizing*) means "anything above the ceiling becomes the ceiling".
We keep the row, we just stop one crazy value from dominating.

In [ ]:
X["person_income"].quantile([0.5, 0.9, 0.99, 0.995, 0.999, 1.0]).to_frame("income")

The 99.5th percentile is a sensible ceiling for income, and 50 years for employment length.
The ceiling for income is **learned from training data** inside our custom step (5.4).

### 5.3 Multicollinearity: drop one of two "twin" features
**VIF (Variance Inflation Factor)** says how well one feature can be predicted from all the others.

| VIF | Meaning |
|---|---|
| below 5 | Fine |
| 5 to 10 | Keep an eye on it |
| above 10 | Strong problem: drop or transform |

VIF needs complete numbers and it is sensitive to outliers, so for this check we fill and cap first.
We also add an intercept (a constant column). That is the standard way to compute VIF.

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

def vif_table(frame):
    with_const = add_constant(frame)
    return pd.DataFrame({
        "feature": frame.columns,
        "VIF": [variance_inflation_factor(with_const.values, i + 1) for i in range(frame.shape[1])],
    })

vif_input = X[num_cols].copy()
vif_input = vif_input.fillna(vif_input.median())
vif_input["person_age"] = vif_input["person_age"].clip(upper=100)
vif_input["person_emp_length"] = vif_input["person_emp_length"].clip(upper=50)
vif_input["person_income"] = vif_input["person_income"].clip(upper=vif_input["person_income"].quantile(0.995))

print("WITH person_age:")
print(vif_table(vif_input).to_string(index=False))
print("\nWITHOUT person_age:")
print(vif_table(vif_input.drop(columns="person_age")).to_string(index=False))

`person_age` and `cb_person_cred_hist_length` share almost the same information (correlation about 0.87).
Notice that two features that overlap this much can show a VIF of only about 4 to 10, so VIF alone may not look dramatic.
The high correlation is what tells us one of them is redundant. After dropping one, the other one's VIF falls to about 1.
Credit history length is the more direct credit signal, so we **drop `person_age`**.

> 💡 **Fairness note for discussion:** in many countries, using age directly in lending decisions is restricted or sensitive.
> Removing it is a *statistical* choice here, but it is also a good starting point for a conversation about fair lending.

### 5.4 Feature engineering: new clues
| New feature | What it tells us |
|---|---|
| `income_percentile` | Where this income stands compared to the training applicants (0 to 1) |
| `income_to_credit_history` | Income per year of credit history |
| `estimated_annual_income` | Income adjusted for years at work (assuming 3% yearly growth) |
| `median_income_emp_group` | Typical income of people with the same employment length |

Two of these (`income_percentile` and `median_income_emp_group`) are **learned from the training data**.
So they need a proper `fit` step, and we write a small class for that.
We save it in its own file, `loan_features.py`, so that a saved model can be loaded later by a dashboard or an app.
(A class that lives only inside a notebook cannot be loaded by other programs.)

In [ ]:
%%writefile loan_features.py
"""Custom feature engineering for the loan-default project.

Everything that must be LEARNED (income cap, income percentiles, median income per
employment length) is learned in fit() from training data, and only APPLIED in transform().
"""
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin

MAX_EMP_YEARS = 50


class LoanFeatureEngineer(BaseEstimator, TransformerMixin):
    def __init__(self, income_cap_quantile=0.995, growth_rate=0.03):
        self.income_cap_quantile = income_cap_quantile
        self.growth_rate = growth_rate

    def fit(self, X, y=None):
        income = X["person_income"]
        self.income_cap_ = float(income.quantile(self.income_cap_quantile))
        capped = income.clip(upper=self.income_cap_)

        # 1001 points that describe the income distribution of the TRAINING data
        self.income_grid_ = np.sort(capped.quantile(np.linspace(0, 1, 1001)).to_numpy())

        # median income for each employment length (in whole years)
        emp = X["person_emp_length"].clip(0, MAX_EMP_YEARS).round()
        self.income_by_emp_ = capped.groupby(emp).median().to_dict()
        self.overall_median_income_ = float(capped.median())
        return self

    def transform(self, X):
        df = X.copy()
        df = df.drop(columns=["person_age"], errors="ignore")      # redundant with credit history length

        # cap impossible / extreme values (the caps were learned in fit)
        df["person_emp_length"] = df["person_emp_length"].clip(0, MAX_EMP_YEARS)
        df["person_income"] = df["person_income"].clip(upper=self.income_cap_)

        # 1. where does this income stand compared to the training applicants?
        position = np.searchsorted(self.income_grid_, df["person_income"].to_numpy(), side="right")
        df["income_percentile"] = np.clip(position / len(self.income_grid_), 0, 1)

        # 2. income per year of credit history
        df["income_to_credit_history"] = df["person_income"] / df["cb_person_cred_hist_length"].clip(lower=1)

        # 3. income adjusted for years at work (assumed yearly growth)
        df["estimated_annual_income"] = df["person_income"] / (1 + self.growth_rate * df["person_emp_length"])

        # 4. typical income of people with the same employment length (from training data)
        emp_key = df["person_emp_length"].round()
        df["median_income_emp_group"] = emp_key.map(self.income_by_emp_).fillna(self.overall_median_income_)
        return df

In [ ]:
from loan_features import LoanFeatureEngineer

# Quick look at what the first two steps produce
# (we fit on all of X here ONLY to look at the output; real modelling fits inside each fold)
look = Pipeline([("impute", imputer), ("engineer", LoanFeatureEngineer())]).fit_transform(X)
look.head()

### 5.5 Encoding: turn words into numbers
| Column | Kind | How we encode |
|---|---|---|
| `loan_grade` | Has an **order** (A better than B ...) | **Ordinal:** A=0, B=1, ... G=6 (we give the order ourselves) |
| `cb_person_default_on_file` | Yes / No | **0 / 1** |
| `loan_intent` | **No order** (medical is not "bigger" than education) | **One-hot:** one 0/1 column per category |
| `person_home_ownership` | **No order** (RENT, MORTGAGE, OWN, OTHER are just labels) | **One-hot** |

> 💡 If we gave RENT=2, MORTGAGE=0, OWN=1, the model would think "rent is twice own". One-hot avoids that false order.

### 5.6 Scaling
Income is around 60,000 and the interest rate is around 10. Without scaling, a linear model may treat income as "more important" just because its numbers are bigger.
`StandardScaler` moves every number to a mean of 0 and a spread of 1.
Tree models (LightGBM, Random Forest) do not need scaling, but it does no harm.
We scale only the continuous numbers, not the 0/1 columns.

### 5.7 Put everything into one recipe

In [ ]:
grade_order = ["A", "B", "C", "D", "E", "F", "G"]
onehot_cols = ["person_home_ownership", "loan_intent"]

numeric_cols = [
    "person_income", "person_emp_length", "loan_amnt", "loan_int_rate",
    "loan_percent_income", "cb_person_cred_hist_length",
    "income_percentile", "income_to_credit_history",
    "estimated_annual_income", "median_income_emp_group",
]

encode_and_scale = ColumnTransformer([
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False), onehot_cols),
    ("grade", OrdinalEncoder(categories=[grade_order], handle_unknown="use_encoded_value", unknown_value=-1), ["loan_grade"]),
    ("default_flag", OrdinalEncoder(categories=[["N", "Y"]], handle_unknown="use_encoded_value", unknown_value=-1), ["cb_person_default_on_file"]),
    ("scale", StandardScaler(), numeric_cols),
], verbose_feature_names_out=False)

preprocessor = Pipeline([
    ("impute", imputer),                       # 5.1  fill missing values
    ("engineer", LoanFeatureEngineer()),       # 5.2 + 5.3 + 5.4  cap, drop age, new features
    ("encode_scale", encode_and_scale),        # 5.5 + 5.6  words -> numbers, then scale
])
preprocessor

In [ ]:
# See the result (fit on all of X only to look at it)
demo_array = clone(preprocessor).fit_transform(X)
demo_names = clone(preprocessor).fit(X)[-1].get_feature_names_out()
pd.DataFrame(demo_array, columns=demo_names).head()

Every column is now a number and there are no missing values. This recipe will be reused for **every model**, always fitted on training data only.

## 6. Split the data and set the "floor"

We hide 20% of the data (**validation set**). The model never trains on it, so it acts like a mini exam.
`stratify=y` keeps the same share of defaulters in both parts.

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score

X_tr, X_val, y_tr, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print("Train:", X_tr.shape, "| Validation:", X_val.shape)
print(f"Default rate  train: {y_tr.mean():.3f}   validation: {y_val.mean():.3f}")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

### The dummy model: the score to beat
A model that ignores all features and just predicts the overall default rate for everyone.
If our real model is not clearly better than this, it is useless.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, roc_auc_score, average_precision_score, classification_report,
                             ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay)

def evaluate(name, y_true, p, threshold=0.5):
    """Print the main scores for predicted probabilities p."""
    pred = (np.asarray(p) >= threshold).astype(int)
    print(f"{name:<26} accuracy={accuracy_score(y_true, pred):.3f}   "
          f"ROC-AUC={roc_auc_score(y_true, p):.3f}   PR-AUC={average_precision_score(y_true, p):.3f}")

dummy = DummyClassifier(strategy="prior").fit(X_tr, y_tr)
p_dummy = dummy.predict_proba(X_val)[:, 1]
evaluate("Dummy (validation)", y_val, p_dummy)

Look at the accuracy: it is high without any intelligence, because most people repay.
The **ROC-AUC is 0.5**, which is a coin flip. This is why we trust AUC more than accuracy here.

## 7. Baseline model: Logistic Regression

> **Why a baseline?** Before buying a sports car, test a bicycle. If the bicycle is good enough, we save money and keep things simple.

Logistic Regression draws a straight-line boundary and turns the result into a probability between 0 and 1.

In [ ]:
from sklearn.linear_model import LogisticRegression

logreg = Pipeline([
    ("prep", clone(preprocessor)),
    ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])
logreg.fit(X_tr, y_tr)

p_tr = logreg.predict_proba(X_tr)[:, 1]
p_val = logreg.predict_proba(X_val)[:, 1]
evaluate("LogReg (train)", y_tr, p_tr)
evaluate("LogReg (validation)", y_val, p_val)

If the train score is much higher than the validation score, the model is **memorising** (overfitting). Here the two are close, which is healthy.

### How to read the results
| Metric | Plain-English question |
|---|---|
| **Accuracy** | How often is the model right? (can fool us when classes are imbalanced) |
| **Precision** | Of the people we *called risky*, how many really defaulted? |
| **Recall** | Of all the people who *really defaulted*, how many did we catch? |
| **ROC-AUC** | If we pick one defaulter and one repayer at random, how often does the model give the defaulter the higher risk? (0.5 = coin flip, 1.0 = perfect) |
| **PR-AUC** | Like AUC, but focused on the rare class (defaulters) |

In [ ]:
pred_val = (p_val >= 0.5).astype(int)
print(classification_report(y_val, pred_val, target_names=["repaid (0)", "defaulted (1)"]))

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
ConfusionMatrixDisplay.from_predictions(y_val, pred_val, display_labels=["repaid", "defaulted"],
                                        cmap="Blues", colorbar=False, ax=axes[0])
axes[0].set_title("Confusion matrix (threshold 0.5)\nrows = real, columns = predicted")
RocCurveDisplay.from_predictions(y_val, p_val, name="LogReg", ax=axes[1])
axes[1].plot([0, 1], [0, 1], "--", color="gray")
axes[1].set_title("ROC curve")
PrecisionRecallDisplay.from_predictions(y_val, p_val, name="LogReg", ax=axes[2])
axes[2].set_title("Precision-Recall curve")
plt.tight_layout()
plt.show()

### The bank's real decision: where do we draw the line?
The model gives a **probability**. We still need a **threshold**: "reject if probability is above ___".
Two mistakes are possible, and they do not cost the same:

* **Missed defaulter:** we approve a loan that is never repaid. We lose money.
* **Rejected good customer:** we say no to someone who would have repaid. We lose a customer.

The numbers below are an **assumption for teaching** (a bad loan costs 5 times more than a lost customer). Change them to see what happens.

In [ ]:
COST_MISSED_DEFAULT = 5     # assumption: approving a loan that defaults
COST_REJECTED_GOOD = 1      # assumption: rejecting someone who would have repaid

def threshold_table(y_true, p, thresholds=np.arange(0.10, 0.91, 0.05)):
    y_true = np.asarray(y_true)
    rows = []
    for t in thresholds:
        flagged = np.asarray(p) >= t
        tp = (flagged & (y_true == 1)).sum()
        fp = (flagged & (y_true == 0)).sum()
        fn = (~flagged & (y_true == 1)).sum()
        rows.append({"threshold": t,
                     "precision": tp / max(tp + fp, 1),
                     "recall": tp / max(tp + fn, 1),
                     "total_cost": fn * COST_MISSED_DEFAULT + fp * COST_REJECTED_GOOD})
    return pd.DataFrame(rows)

def plot_thresholds(table, title):
    best = table.loc[table["total_cost"].idxmin()]
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(table["threshold"], table["precision"], label="precision")
    ax.plot(table["threshold"], table["recall"], label="recall")
    ax.set_xlabel("threshold"); ax.set_ylabel("precision / recall")
    ax2 = ax.twinx()
    ax2.plot(table["threshold"], table["total_cost"], color="red", ls="--", label="total cost")
    ax2.set_ylabel("total cost", color="red"); ax2.grid(False)
    ax.axvline(best["threshold"], color="gray", ls=":")
    ax.set_title(f"{title}\nlowest cost at threshold {best['threshold']:.2f}")
    handles = ax.get_lines()[:2] + ax2.get_lines()
    ax2.legend(handles, [h.get_label() for h in handles], loc="upper center", ncol=3, frameon=True)
    plt.show()
    return best["threshold"]

tt = threshold_table(y_val, p_val)
best_t_logreg = plot_thresholds(tt, "Logistic Regression: precision, recall and cost vs threshold")
tt.round(3).iloc[::2]

> 🤔 **Discuss:** a *low* threshold flags many people (high recall, low precision). A *high* threshold flags few (the opposite).
> Which end would your bank choose, and why? Try `COST_MISSED_DEFAULT = 1` and watch the best threshold move.

### Cross-validation: five exams instead of one
One validation split can be lucky or unlucky. **Cross-validation** trains and tests five times on different parts and averages.
Because the preprocessing is inside the pipeline, each round learns its medians, caps and scalers from its own training part only (no leakage).

In [ ]:
cv_logreg = cross_val_score(logreg, X, y, cv=cv, scoring="roc_auc")
print("ROC-AUC per fold:", cv_logreg.round(4))
print(f"Mean: {cv_logreg.mean():.4f}  (+/- {cv_logreg.std():.4f})")

### What did the linear model learn?
Because the numbers are scaled, the size of each coefficient shows how strongly a feature pushes the prediction.
Positive = more default risk, negative = less. (When two features are related, they share the credit, so do not over-interpret single bars.)

In [ ]:
names_lr = logreg.named_steps["prep"][-1].get_feature_names_out()
coefs = pd.Series(logreg.named_steps["model"].coef_[0], index=names_lr).sort_values()

fig, ax = plt.subplots(figsize=(9, 6))
coefs.plot.barh(color=np.where(coefs > 0, "#d9534f", "#4a78c2"), ax=ax)
ax.axvline(0, color="black", lw=0.8)
ax.set_title("Logistic Regression coefficients (red = pushes towards default)")
plt.tight_layout()
plt.show()

## 8. Which features matter? (Random Forest)

A Random Forest is **100 decision trees that vote**. Each tree asks questions like "is `loan_percent_income` above 0.3?".
Afterwards we ask: *which questions were the most useful across all trees?*

Go back to your guess from Step 2. How many of your three columns made the top?

In [ ]:
from sklearn.ensemble import RandomForestClassifier

prep_rf = clone(preprocessor).fit(X_tr)            # learn the recipe on training rows only
names_rf = prep_rf[-1].get_feature_names_out()
Xt_tr = pd.DataFrame(prep_rf.transform(X_tr), columns=names_rf, index=X_tr.index)
Xt_val = pd.DataFrame(prep_rf.transform(X_val), columns=names_rf, index=X_val.index)

rf = RandomForestClassifier(n_estimators=100, min_samples_leaf=3, n_jobs=-1, random_state=RANDOM_STATE)
rf.fit(Xt_tr, y_tr)
evaluate("Random Forest (validation)", y_val, rf.predict_proba(Xt_val)[:, 1])

importance = pd.Series(rf.feature_importances_, index=names_rf).sort_values()
fig, ax = plt.subplots(figsize=(10, 6))
importance.plot.barh(color="royalblue", ax=ax)
ax.set_title("Feature importance (Random Forest)")
ax.set_xlabel("importance")
plt.tight_layout()
plt.show()

> 💡 **Caution:** this importance tells us what the model *uses*, not what *causes* default.
> It also tends to favour continuous numbers with many possible values. For a deeper check you can try *permutation importance* (an exercise at the end).
>
> We keep all features: tree models handle weak ones without trouble.

## 9. A stronger model: LightGBM with Stratified K-Fold

**Boosting analogy:** a student takes a test, then studies *only the questions they got wrong*, retakes it, and repeats this hundreds of times.
Each new small tree focuses on the mistakes of the trees before it.

| Setting | Plain meaning |
|---|---|
| `n_estimators` | Maximum number of trees (study rounds) |
| `learning_rate` | Size of each correction. Small steps are safer but slower |
| `num_leaves` | How complex each tree can be |
| **early stopping** | "Stop studying when the validation score stops improving" |
| **Stratified K-Fold** | 5 rounds; each round the share of defaulters stays the same |

The other settings are tuning details: sampling rows and columns to reduce overfitting.

In each fold we: (1) learn the recipe from the fold's training part, (2) train LightGBM, (3) score the fold's hidden part.
All the hidden predictions together are called **out-of-fold (OOF)** predictions. Every row is predicted by a model that never saw it.

In [ ]:
import lightgbm as lgb

# Newest LightGBM versions print a harmless notice about `eval_set`; it still works in every version
warnings.filterwarnings("ignore", message=".*eval_set.*")

lgb_params = dict(
    objective="binary",
    metric="auc",
    n_estimators=3000,            # upper limit: early stopping decides the real number
    learning_rate=0.0323,
    num_leaves=24,
    max_depth=15,
    min_child_samples=25,         # every leaf needs at least 25 rows
    colsample_bytree=0.6236,      # each tree sees ~62% of the columns
    subsample=0.9597,             # each tree sees ~96% of the rows
    subsample_freq=3,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=-1,
)

oof_pred = np.zeros(len(X))
fold_auc, best_iterations, fold_importance = [], [], []

for fold, (tr_idx, va_idx) in enumerate(cv.split(X, y), start=1):
    X_f_tr, X_f_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_f_tr, y_f_va = y.iloc[tr_idx], y.iloc[va_idx]

    prep = clone(preprocessor).fit(X_f_tr)                       # learn recipe on this fold's training part only
    names = prep[-1].get_feature_names_out()
    Z_tr = pd.DataFrame(prep.transform(X_f_tr), columns=names)
    Z_va = pd.DataFrame(prep.transform(X_f_va), columns=names)

    model = lgb.LGBMClassifier(**lgb_params)
    model.fit(Z_tr, y_f_tr,
              eval_set=[(Z_va, y_f_va)],
              callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)])

    oof_pred[va_idx] = model.predict_proba(Z_va)[:, 1]
    fold_auc.append(roc_auc_score(y_f_va, oof_pred[va_idx]))
    best_iterations.append(model.best_iteration_)
    fold_importance.append(pd.Series(model.booster_.feature_importance(importance_type="gain"), index=names))
    print(f"Fold {fold}/5   AUC = {fold_auc[-1]:.4f}   trees used = {model.best_iteration_}")

print(f"\nMean fold AUC: {np.mean(fold_auc):.4f} (+/- {np.std(fold_auc):.4f})")
print(f"Overall OOF AUC: {roc_auc_score(y, oof_pred):.4f}")

> 💡 **Small honesty note:** early stopping looks at the validation part of each fold to choose the number of trees, so these scores are very slightly optimistic. The size of that effect is small, but it is good to know.

### Compare with the baseline

In [ ]:
comparison = pd.DataFrame({
    "model": ["Dummy (always the average)", "Logistic Regression", "LightGBM"],
    "5-fold ROC-AUC": [0.5, cv_logreg.mean(), np.mean(fold_auc)],
})
comparison

### Look at the LightGBM results

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

RocCurveDisplay.from_predictions(y, oof_pred, name="LightGBM (OOF)", ax=axes[0])
axes[0].plot([0, 1], [0, 1], "--", color="gray")
axes[0].set_title("ROC curve: higher AUC = better ranking")

PrecisionRecallDisplay.from_predictions(y, oof_pred, name="LightGBM (OOF)", ax=axes[1])
axes[1].set_title("Precision-Recall curve")

imp = pd.concat(fold_importance, axis=1).fillna(0).mean(axis=1).sort_values()
imp = imp / imp.sum()
imp.plot.barh(color="seagreen", ax=axes[2])
axes[2].set_title("Feature importance (average over 5 folds)")
plt.tight_layout()
plt.show()

In [ ]:
tt_lgb = threshold_table(y, oof_pred)
best_t = plot_thresholds(tt_lgb, "LightGBM: precision, recall and cost vs threshold")
print(f"Chosen threshold (lowest assumed cost): {best_t:.2f}")

fig, ax = plt.subplots(figsize=(5.5, 4.5))
ConfusionMatrixDisplay.from_predictions(y, (oof_pred >= best_t).astype(int),
                                        display_labels=["repaid", "defaulted"], cmap="Blues", colorbar=False, ax=ax)
ax.set_title(f"Confusion matrix at threshold {best_t:.2f}\nrows = real, columns = predicted")
plt.show()

> 🤔 **Discuss:** compare this confusion matrix with the logistic regression one. How many more defaulters do we catch? What do we pay for it (more rejected good customers)?

## 10. Final model and submission

For the final model we train on **all** the labelled data (more data = better estimates).
We do not have a validation part left for early stopping, so we reuse what the folds told us:
the average number of trees, plus 10% because we now have about 25% more data than in each fold.

In [ ]:
n_trees_final = int(np.mean(best_iterations) * 1.1)
print("Trees in the final model:", n_trees_final)

final_pipeline = Pipeline([
    ("prep", clone(preprocessor)),
    ("model", lgb.LGBMClassifier(**{**lgb_params, "n_estimators": n_trees_final})),
])
final_pipeline.fit(X, y)

In [ ]:
test_probs = final_pipeline.predict_proba(test)[:, 1]

submission = pd.DataFrame({"id": test_ids, "loan_status": test_probs})

# Safety checks before sending anything
assert list(submission.columns) == list(sample_submission.columns), "columns differ from the sample"
assert len(submission) == len(sample_submission), "row count differs from the sample"
assert (submission["id"].values == sample_submission["id"].values).all(), "ids are not in the same order"
assert submission["loan_status"].between(0, 1).all() and submission["loan_status"].notna().all()

submission.to_csv("lgbm_submission.csv", index=False)
print("Saved lgbm_submission.csv")
print(f"Average predicted default probability on test: {test_probs.mean():.3f}   (training default rate: {y.mean():.3f})")
submission.head()

The two averages should be in the same area. If they are far apart, something is wrong (a different kind of applicants, or a bug).

## 11. Save the model and use it on one new applicant

`joblib.dump` saves the **whole pipeline** (recipe + model) in one file. A dashboard or an app only needs to:
1. load the file,
2. build a table with **one row** of raw applicant data,
3. call `predict_proba`.

No manual preprocessing, because it is all inside the pipeline.

In [ ]:
import joblib

joblib.dump(final_pipeline, "model.pkl")
loaded = joblib.load("model.pkl")

# Meet Sara: 25 years old, income $40,000, rents her home, wants a $10,000 personal loan
sara = pd.DataFrame([{
    "person_age": 25,
    "person_income": 40000,
    "person_home_ownership": "RENT",
    "person_emp_length": 2.0,
    "loan_intent": "PERSONAL",
    "loan_grade": "C",
    "loan_amnt": 10000,
    "loan_int_rate": 13.5,
    "loan_percent_income": 0.25,
    "cb_person_default_on_file": "N",
    "cb_person_cred_hist_length": 3,
}])[X.columns]                       # same column order as the training data

p_sara = loaded.predict_proba(sara)[0, 1]
print(f"Sara's estimated probability of default: {p_sara:.1%}")
print("Decision at our threshold:", "send for review / decline" if p_sara >= best_t else "approve")

print("\nVersions (the saved model needs the same ones to load safely):")
print("  scikit-learn", sklearn.__version__, "| lightgbm", lgb.__version__, "| pandas", pd.__version__)

> 💡 **Deployment notes**
> * Keep `loan_features.py` **next to** `model.pkl`: the pickle needs that class to load.
> * Use the same library versions where you load the model.
> * The percentile feature is computed against the **training** income distribution, so scoring a single person works correctly. (A version that ranks incomes *inside the new data* would give every single applicant the same value.)

## 12. Recap

**Understand &rarr; Clean &rarr; Prepare &rarr; Baseline &rarr; Improve &rarr; Package**

1. Check the target balance first: it decides which metrics we can trust.
2. Read **rates**, not raw counts, when comparing groups.
3. Everything the model learns must be learned from training data only. A Pipeline enforces this.
4. Always compare against a dummy and a simple baseline before a complex model.
5. A probability is not a decision. The threshold depends on the cost of each mistake.
6. Save the whole pipeline, not just the model.

## Exercises for students
1. Change `COST_MISSED_DEFAULT` to 1, then 10. How does the best threshold move? Why?
2. Remove `loan_grade` from the data. How much does the AUC drop? What does that tell you?
3. Keep `person_age` (remove it from `LoanFeatureEngineer.transform`). Does the AUC change? Would you keep it, knowing the fairness question?
4. Add your own feature (for example `loan_amnt * loan_int_rate / person_income`) inside `LoanFeatureEngineer`. Does the CV score improve?
5. Compute **permutation importance** on the validation set (`sklearn.inspection.permutation_importance`) and compare it with the Random Forest chart.